In [1]:
import pandas as pd

movies  = pd.read_csv("data/raw/movies.csv")
ratings = pd.read_csv("data/raw/ratings.csv")
tags    = pd.read_csv("data/raw/tags.csv")
links   = pd.read_csv("data/raw/links.csv")

for nome, df in [("movies", movies), ("ratings", ratings), ("tags", tags), ("links", links)]:
    print(f"\n===== {nome} =====")
    print("Linhas, colunas:", df.shape)
    print("Colunas:", list(df.columns))
    print("Nulos por coluna:\n", df.isnull().sum())
    print(df.head(3))


===== movies =====
Linhas, colunas: (9742, 3)
Colunas: ['movieId', 'title', 'genres']
Nulos por coluna:
 movieId    0
title      0
genres     0
dtype: int64
   movieId                    title  \
0        1         Toy Story (1995)   
1        2           Jumanji (1995)   
2        3  Grumpier Old Men (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  

===== ratings =====
Linhas, colunas: (100836, 4)
Colunas: ['userId', 'movieId', 'rating', 'timestamp']
Nulos por coluna:
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224

===== tags =====
Linhas, colunas: (3683, 4)
Colunas: ['userId', 'movieId', 'tag', 'timestamp']
Nulos por coluna:
 userId       0
movieId    

Como os ratings tem 100836 entradas, e o enunciado pede apenas 100000, vamos tirar 836 das entradas de forma aleatória mas que seja reproduzivel no futuro.

In [2]:
ratings = ratings.sample(n=100000, random_state=42).reset_index(drop=True)

print("Ratings depois de remover:", ratings.shape)
print(ratings.head(3))

Ratings depois de remover: (100000, 4)
   userId  movieId  rating   timestamp
0     432    77866     4.5  1335139641
1     288      474     3.0   978465565
2     599     4351     3.0  1498524542


Antes de limpar os dados, verificamos se há valores em falta no `ratings`, `tags` e `links` (o `movies` já foi confirmado sem nulos no output inicial).

In [4]:
print(ratings.isnull().sum())
print(tags.isnull().sum())
print(links.isnull().sum())

userId       0
movieId      0
rating       0
timestamp    0
dtype: int64
userId       0
movieId      0
tag          0
timestamp    0
dtype: int64
movieId    0
imdbId     0
tmdbId     8
dtype: int64


Os 8 filmes sem `tmdbId` são mantidos, com o valor vazio (NULL), para não perdermos nem os filmes nem as suas notas. Convertemos a coluna para inteiro, porque os valores em falta tinham transformado em decimal.

In [5]:
links["tmdbId"] = links["tmdbId"].astype("Int64")
print(links.head(3))

   movieId  imdbId  tmdbId
0        1  114709     862
1        2  113497    8844
2        3  113228   15602


Verificamos se há duplicados: o `movieId` vai ser a chave primária dos filmes, por isso não pode repetir; e cada utilizador só deve ter uma nota por filme.

In [6]:
print("movieId repetidos:", movies["movieId"].duplicated().sum())
print("Notas repetidas (mesmo user e filme):", ratings.duplicated(subset=["userId", "movieId"]).sum())

movieId repetidos: 0
Notas repetidas (mesmo user e filme): 0


Alguns filmes não têm género: em vez de um valor vazio, têm o texto "(no genres listed)". Precisamos de ver quantos são.

In [9]:
sem_genero = movies[movies["genres"] == "(no genres listed)"]
print("Filmes sem género:", len(sem_genero))
print(sem_genero.head())

Filmes sem género: 34
      movieId                                              title  \
8517   114335                                  La cravate (1957)   
8684   122888                                     Ben-hur (2016)   
8687   122896  Pirates of the Caribbean: Dead Men Tell No Tal...   
8782   129250                                  Superfast! (2015)   
8836   132084                                Let It Be Me (1995)   

                  genres  
8517  (no genres listed)  
8684  (no genres listed)  
8687  (no genres listed)  
8782  (no genres listed)  
8836  (no genres listed)  


Existem 34 filmes com "(no genres listed)". Mantemos estes filmes, mas sem nenhum género associado, para não criar um género falso. Isto vai ser tratado ao separar os géneros.

O ano do filme vem dentro do título, por exemplo "Toy Story (1995)". Antes de o separar numa coluna própria, verificamos se há títulos sem ano no fim.

In [10]:
sem_ano = movies[~movies["title"].str.strip().str.contains(r"\(\d{4}\)$")]
print("Títulos sem ano:", len(sem_ano))
print(sem_ano)

Títulos sem ano: 13
      movieId                                              title  \
6059    40697                                          Babylon 5   
9031   140956                                   Ready Player One   
9091   143410                                         Hyena Road   
9138   147250  The Adventures of Sherlock Holmes and Doctor W...   
9179   149334                                  Nocturnal Animals   
9259   156605                                           Paterson   
9367   162414                                          Moonlight   
9448   167570                                             The OA   
9514   171495                                             Cosmos   
9515   171631                            Maria Bamford: Old Baby   
9518   171749                  Death Note: Desu nôto (2006–2007)   
9525   171891                                  Generation Iron 2   
9611   176601                                       Black Mirror   

                      genre

A decisão ficou separar o ano do título numa coluna `year`. Os 13 títulos sem ano ficam com o ano vazio (NULL).

In [11]:
movies["title"] = movies["title"].str.strip()
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)$")[0].astype("Int64")
movies["title"] = movies["title"].str.replace(r"\s*\(\d{4}\)$", "", regex=True)

print(movies[["movieId", "title", "year"]].head(3))
print("Filmes sem ano:", movies["year"].isnull().sum())

   movieId             title  year
0        1         Toy Story  1995
1        2           Jumanji  1995
2        3  Grumpier Old Men  1995
Filmes sem ano: 13


O `timestamp` está em segundos desde 1970. Convertemo-lo para data e hora, no `ratings` e no `tags`, para ser legível e permitir queries por data

In [12]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
tags["timestamp"] = pd.to_datetime(tags["timestamp"], unit="s")

print(ratings.head(3))
print(tags.head(3))

   userId  movieId  rating           timestamp
0     432    77866     4.5 2012-04-23 00:07:21
1     288      474     3.0 2001-01-02 19:59:25
2     599     4351     3.0 2017-06-27 00:49:02
   userId  movieId              tag           timestamp
0       2    60756            funny 2015-10-24 19:29:54
1       2    60756  Highly quotable 2015-10-24 19:29:56
2       2    60756     will ferrell 2015-10-24 19:29:52


Transformamos o texto dos géneros (separados por "|") numa lista. Os filmes com "(no genres listed)" ficam com a lista vazia.

In [13]:
movies["genres"] = movies["genres"].replace("(no genres listed)", "")
movies["genres"] = movies["genres"].str.split("|").apply(lambda g: [x for x in g if x != ""])

print(movies[["title", "genres"]].head(3))
print("Filmes com lista vazia:", (movies["genres"].str.len() == 0).sum())

              title                                             genres
0         Toy Story  [Adventure, Animation, Children, Comedy, Fantasy]
1           Jumanji                     [Adventure, Children, Fantasy]
2  Grumpier Old Men                                  [Comedy, Romance]
Filmes com lista vazia: 34


Para o MySQL, a relação filme-género é many-to-many. Como já sabemos MySQL nao suporta lista dentro de colunas, entao o que fazemos é a partir da lista de géneros, criamos duas tabelas novas: `genres` (um id por género) e `movie_genres` (que liga cada filme aos seus géneros). O `movies` com a lista mantém-se para o MongoDB.

In [15]:
movie_genres = movies[["movieId", "genres"]].explode("genres").dropna()
movie_genres = movie_genres.rename(columns={"genres": "genre"})

genres = pd.DataFrame({"genre": sorted(movie_genres["genre"].unique())})
genres["genreId"] = range(1, len(genres) + 1)

movie_genres = movie_genres.merge(genres, on="genre")[["movieId", "genreId"]]

print(genres)
print(movie_genres.head())
print("Ligações filme-género:", len(movie_genres))

          genre  genreId
0        Action        1
1     Adventure        2
2     Animation        3
3      Children        4
4        Comedy        5
5         Crime        6
6   Documentary        7
7         Drama        8
8       Fantasy        9
9     Film-Noir       10
10       Horror       11
11         IMAX       12
12      Musical       13
13      Mystery       14
14      Romance       15
15       Sci-Fi       16
16     Thriller       17
17          War       18
18      Western       19
   movieId  genreId
0        1        2
1        1        3
2        1        4
3        1        5
4        1        9
Ligações filme-género: 22050


Como cada filme tem apenas um `imdbId` e um `tmdbId` (one-to.one), juntamos o `links` ao `movies` em vez de criar uma tabela à parte.

In [16]:
movies = movies.merge(links, on="movieId", how="left")
print(movies.head(3))

   movieId             title  \
0        1         Toy Story   
1        2           Jumanji   
2        3  Grumpier Old Men   

                                              genres  year  imdbId  tmdbId  
0  [Adventure, Animation, Children, Comedy, Fantasy]  1995  114709     862  
1                     [Adventure, Children, Fantasy]  1995  113497    8844  
2                                  [Comedy, Romance]  1995  113228   15602  


O dataset não tem ficheiro de users. Criamos a tabela `users` com todos os `userId` que aparecem no `ratings` e no `tags`, para que todas as notas e tags apontem para um utilizador que ja existe.

In [17]:
user_ids = pd.concat([ratings["userId"], tags["userId"]]).unique()
users = pd.DataFrame({"userId": sorted(user_ids)})

print(users.head())
print("Utilizadores:", len(users))

   userId
0       1
1       2
2       3
3       4
4       5
Utilizadores: 610


Agora antes de gravar temos de verificar se tudo está a apontar corretamente: todas as notas e tags têm de apontar para filmes e utilizadores que existem.

In [18]:
print("Ratings com filme inexistente:", (~ratings["movieId"].isin(movies["movieId"])).sum())
print("Tags com filme inexistente:", (~tags["movieId"].isin(movies["movieId"])).sum())
print("Ratings com user inexistente:", (~ratings["userId"].isin(users["userId"])).sum())
print("Tags com user inexistente:", (~tags["userId"].isin(users["userId"])).sum())

Ratings com filme inexistente: 0
Tags com filme inexistente: 0
Ratings com user inexistente: 0
Tags com user inexistente: 0


Visualizar como fica cada tabela

In [19]:
for nome, df in [("movies", movies), ("genres", genres), ("movie_genres", movie_genres),
                 ("users", users), ("ratings", ratings), ("tags", tags)]:
    print(f"\n===== {nome} ===== {df.shape}")
    print(df.head(3))


===== movies ===== (9742, 6)
   movieId             title  \
0        1         Toy Story   
1        2           Jumanji   
2        3  Grumpier Old Men   

                                              genres  year  imdbId  tmdbId  
0  [Adventure, Animation, Children, Comedy, Fantasy]  1995  114709     862  
1                     [Adventure, Children, Fantasy]  1995  113497    8844  
2                                  [Comedy, Romance]  1995  113228   15602  

===== genres ===== (19, 2)
       genre  genreId
0     Action        1
1  Adventure        2
2  Animation        3

===== movie_genres ===== (22050, 2)
   movieId  genreId
0        1        2
1        1        3
2        1        4

===== users ===== (610, 1)
   userId
0       1
1       2
2       3

===== ratings ===== (100000, 4)
   userId  movieId  rating           timestamp
0     432    77866     4.5 2012-04-23 00:07:21
1     288      474     3.0 2001-01-02 19:59:25
2     599     4351     3.0 2017-06-27 00:49:02

===== tags

Guardamos os dados limpos em `data/processed/`. Para o MySQL, cada tabela num CSV (o `movies` sem a lista de géneros). Para o MongoDB, o `movies` com a lista de géneros num JSON, porque o CSV não guarda listas.

In [20]:
import os
os.makedirs("data/processed", exist_ok=True)

# MySQL
movies.drop(columns=["genres"]).to_csv("data/processed/movies.csv", index=False)
genres.to_csv("data/processed/genres.csv", index=False)
movie_genres.to_csv("data/processed/movie_genres.csv", index=False)
users.to_csv("data/processed/users.csv", index=False)
ratings.to_csv("data/processed/ratings.csv", index=False)
tags.to_csv("data/processed/tags.csv", index=False)

# MongoDB
movies.to_json("data/processed/movies_mongo.json", orient="records", indent=2)

print(os.listdir("data/processed"))

['users.csv', 'tags.csv', 'movies_mongo.json', 'genres.csv', 'ratings.csv', 'movie_genres.csv', 'movies.csv']
